# Day 41 · 计费与合规

**配套讲义**: [`days/day-41.md`](../days/day-41.md) ｜ **本地可跑，不需要 GPU**

实现订阅计划（免费试用 / 按会话量计费）+ 用量上报；补齐 3 个 GDPR webhook；走完一次订阅流程，并能说清「按用量计费」在 Shopify 上怎么对账。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w7.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys
print("python:", sys.version.split()[0])
for m in ("numpy", "PIL", "yaml", "pandas"):
    try:
        mod = __import__(m)
        print(f"  {m:7s} {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        print(f"  {m:7s} ❌ 缺 → pip install {m}")
print("\n→ 本机没 GPU 不影响今天：今天只用纯 Python / numpy")

## 1. 看数据库 schema

In [ ]:
from pathlib import Path
text = Path("../src/shopify/models.py").read_text()
i = text.find("SCHEMA")
print(text[i:i + 1800])

## 2. 幂等上报亲手验

In [ ]:
import sys; sys.path.insert(0, "..")
from src.shopify.billing import BillingLedger

ledger = BillingLedger()
for i in range(3):
    ledger.record(session_id="sess_001", amount=0.05)   # 同一个会话上报 3 次

print("台账条目数:", len(ledger.entries) if hasattr(ledger, "entries") else "（看实际属性名）")
print("→ 必须是 1，不是 3")

## 3. 单位经济：这个生意能不能做

In [ ]:
GPU_HOURLY = 1.88          # RTX 4090 ¥/h
SESSIONS_PER_HOUR = 300     # 估算：单卡并发 4，单次 5s
PLAN_PRICE = 99             # 标准版月费
SESSIONS_INCLUDED = 2000

gpu_per_session = GPU_HOURLY / SESSIONS_PER_HOUR
print(f"单会话 GPU 成本 ≈ ¥{gpu_per_session:.4f}")
print(f"套餐含 {SESSIONS_INCLUDED} 次会话 → GPU 成本 ¥{gpu_per_session*SESSIONS_INCLUDED:.2f}")
print(f"套餐价 ¥{PLAN_PRICE} → 毛利 ¥{PLAN_PRICE - gpu_per_session*SESSIONS_INCLUDED:.2f}")
print("\n（还没算 Shopify 抽成、服务器、人力 —— W8 Day 43 会做完整模型）")

## 验收清单

- [ ] 能走完一次订阅流程（创建 → 授权确认 → 状态可查）
- [ ] 用量上报幂等（同一会话上报 3 次只记 1 条）
- [ ] **本地台账和上报记录能对上**（这是对账的基础）
- [ ] 3 个 GDPR webhook 都实现并自检通过
- [ ] 能说清「按用量计费」的对账流程：Shopify 那边能看到什么、你这边要记什么

**卡住了？** 回看 [`days/day-41.md`](../days/day-41.md) 第五节「容易踩的坑」。

> **明天**：`days/day-42.md` —— 部署上线，W7 收官